# Grid Up Datathon — 02 · Baseline

Amaç: **en hızlı geçerli submission**. Optimize etmeden önce çalışan bir uçtan
uca hattın olsun. İlk gün hedefi tek bir sayı: leaderboard'da bir skor.

Sıra: fold'lar → feature → eğit → doğrula → yaz.

In [ ]:
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from gridup import (
    conditional_quantile_from_hurdle, cross_validate, fit_quantile_ladder,
    fit_two_stage, make_model_zoo, read_any, set_global_seed,
    sweep_count_objectives, tune_with_optuna, write_submission, zero_baseline_score,
)
from gridup.compat import categorical_columns
from gridup.ensemble import hill_climb_weights, prune_by_correlation, stack_oof
from gridup.experiment import ExperimentLog, ExperimentRecord
from gridup.features import (
    add_calendar_features, add_frequency_encoding, add_lag_features,
    add_neighbour_target_lag, add_physical_derivatives, add_regional_aggregates,
    nearest_neighbours, shared_origin,
)
from gridup.metrics import inverse_log_transform, log_transform_target
from gridup.models import starter_params
from gridup.refit import estimate_full_data_rounds, extract_best_iterations, multi_seed_refit
from gridup.selection import null_importance_filter, shap_backward_selection
from gridup.validation import adversarial_validation, build_splitter, purged_time_series_split

set_global_seed(42)

DATA_DIR = Path("/kaggle/input/GRID-UP-YARISMA-SLUG") if IS_KAGGLE else Path("../data/raw")
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("../submissions")

TARGET = "HEDEF_KOLON"     # TODO
ID_COLUMN = "id"           # TODO
TIME_COLUMN = None         # TODO
GROUP_COLUMN = None        # TODO
METRIC = "rmse"            # TODO — yarışmanın resmi metriği (2024'te MAE idi!)
TASK = "regression"        # regression | binary | multiclass
LOG_TARGET = False         # metrik RMSLE ise veya hedef çok çarpıksa True

# TAHMİN UFKU — en pahalı sessiz hatanın kaynağı.
# Test ileriideki bir BLOK ise (ör. bir sonraki ay), o bloğun son gününü
# tahmin ederken elindeki en taze veri blok uzunluğu kadar eskidir.
# shift(1) ile hesaplanan lag'ler CV'de harika görünür, private LB'de çöker.
# Veri geldiğinde: HORIZON = (test.tarih.max() - test.tarih.min()).days + 1
HORIZON = 1                # TODO

In [ ]:
train = read_any(DATA_DIR / "train.csv")
test  = read_any(DATA_DIR / "test.csv")
print(train.shape, test.shape)

## 1 · Fold'lar — feature üretmeden ÖNCE

Sıra önemli: hedef kodlama fold'lara ihtiyaç duyar. Fold'ları önce sabitle ki
tüm deneyler **aynı bölmeler** üzerinde karşılaştırılabilir olsun.

In [ ]:
if TIME_COLUMN:
    train[TIME_COLUMN] = pd.to_datetime(train[TIME_COLUMN])
    test[TIME_COLUMN] = pd.to_datetime(test[TIME_COLUMN])
    HORIZON = int((test[TIME_COLUMN].max() - test[TIME_COLUMN].min()).days) + 1
    print(f"Tahmin ufku (test blok uzunluğu): {HORIZON} gün")
    # Ambargo: en uzun kayan pencerenden BÜYÜK olmalı (zorunlu parametre)
    folds = purged_time_series_split(train[TIME_COLUMN], n_splits=5,
                                     embargo=pd.Timedelta(days=30))
elif GROUP_COLUMN:
    splitter = build_splitter("GroupKFold", n_splits=5)
    folds = list(splitter.split(train, groups=train[GROUP_COLUMN]))
else:
    scheme = "StratifiedKFold" if TASK != "regression" else "KFold"
    splitter = build_splitter(scheme, n_splits=5, seed=42)
    folds = list(splitter.split(train, train[TARGET] if TASK != "regression" else None))

for i, (tr, va) in enumerate(folds, 1):
    print(f"fold {i}: train={len(tr):>8,}  valid={len(va):>8,}")

## 2 · Feature'lar

**Kural:** train ve test'e *aynı* fonksiyon uygulanır. Ayrı kod yolları,
eğitim/servis uyumsuzluğunun bir numaralı kaynağıdır.

In [ ]:
# ORTAK zaman başlangıcı: train ve test için ayrı ayrı hesaplanırsa test'in
# gün sayacı yeniden 0'dan başlar ve model test'i train'in geçmişi sanır.
# Bu hata lokal CV'de GÖRÜNMEZ — sadece leaderboard çöker.
ORIGIN = shared_origin(train, test, time_column=TIME_COLUMN) if TIME_COLUMN else None

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Train ve test'e aynı dönüşümleri uygular. Girdiyi değiştirmez."""
    out = frame.copy()
    if TIME_COLUMN:
        out = add_calendar_features(out, TIME_COLUMN, include_year=False, origin=ORIGIN)
    # categorical_columns: pandas 2.x ve 3.x'te de doğru çalışır.
    # Düz `dtype == object` kontrolü pandas 3.0'da metin kolonlarını KAÇIRIR.
    categorical = categorical_columns(out)
    if categorical:
        out = add_frequency_encoding(out, categorical[:12])
    return out

train_features = build_features(train)
test_features = build_features(test)

drop = {TARGET, ID_COLUMN, TIME_COLUMN} - {None}
FEATURES = [c for c in train_features.columns
            if c not in drop and c in test_features.columns]
print(f"{len(FEATURES)} feature")

## 3 · Eğit

In [ ]:
y = train_features[TARGET].to_numpy()
if LOG_TARGET:
    y = log_transform_target(y)

params = starter_params("lightgbm", TASK)

result = cross_validate(
    train_features[FEATURES], y, folds,
    kind="lightgbm", task_type=TASK, metric=METRIC,
    params=params, test=test_features[FEATURES],
)

print(result.summary())

## 4 · Submission

`write_submission` yazmadan önce doğrular: NaN, sonsuz, eksik ID, sabit tahmin,
negatif değer. Kaggle'ın "Submission Scoring Error" mesajı sana hiçbir şey söylemez.

In [ ]:
predictions = result.test_predictions
if LOG_TARGET:
    predictions = inverse_log_transform(predictions)

path = write_submission(
    test_features[ID_COLUMN].to_numpy(),
    predictions,
    OUT_DIR / "baseline_lgbm.csv",
    id_column=ID_COLUMN,
    target_column=TARGET,
)

## 5 · Deney defterine yaz

Submission gönderdikten **sonra** leaderboard skorunu geri yaz:

```python
log.record_lb("baseline_lgbm", 12.3456)
print(log.cv_lb_correlation())
```

CV–LB korelasyonu bu yarışmanın en önemli tek sayısıdır. r > 0.8 ise CV'ne
güven; r < 0.5 ise CV şemanı düzeltmeden devam etme.

In [ ]:
log = ExperimentLog(OUT_DIR.parent / "experiments" / "deneyler.jsonl")

log.add(ExperimentRecord(
    name="baseline_lgbm",
    cv_score=result.overall_score,
    metric=METRIC,
    model_kind="lightgbm",
    n_features=len(FEATURES),
    fold_scores=result.fold_scores,
    notes="baseline: takvim + frekans kodlama",
    submission_path=str(path),
))

log.leaderboard()

## Sonraki adımlar

Sıra önemli — her adımdan sonra CV'yi ölçüp deftere yazın.

**1 · Kayma kontrolü**
```python
sonuc = adversarial_validation(train[FEATURES], test[FEATURES])
print(sonuc["auc"], sonuc["verdict"])   # AUC > 0.8 ise ayrıştıran feature'ı çıkar
```

**2 · Ufuk-farkındalıklı lag/rolling** — en güçlü aile
```python
out = add_lag_features(out, TARGET, [1, 7, 28], time_column=TIME_COLUMN,
                       group_columns=[GROUP_COLUMN], horizon=HORIZON)
```

**3 · Hazır harici veri** (indirilmiş, `data/` altında)
```python
hava = pd.read_parquet("../data/external/hava_gunluk.parquet")
ilceler = pd.read_parquet("../data/reference/ilceler_gdz_adm.parquet")
komsu = nearest_neighbours(ilceler, key_column="ilce_key",
                           latitude_column="lat", longitude_column="lon", k=3)
out = add_neighbour_target_lag(out, komsu, key_column="ilce_key",
                               time_column=TIME_COLUMN, target_column=TARGET,
                               horizon=HORIZON)
```
Havada **ortalama değil `max` ve quantile** kullanın — hasarı rüzgârın ortalaması
değil tepesi yapar: `add_regional_aggregates`, `add_physical_derivatives`.

**4 · Sayım hedefiyse objective süpürmesi**
```python
zoo = sweep_count_objectives(train[FEATURES], y, folds, metric=METRIC)
print(zoo.leaderboard())   # poisson / tweedie / mae / l2 aynı fold'larda
```

**5 · Sıfır oranı > %40 ise iki aşamalı model**
```python
print(zero_baseline_score(y, metric="mae"))   # önce bunu geçtiğini gör
sonuc = fit_two_stage(train[FEATURES], y, folds, metric=METRIC)
```
Metrik MAE ise `q* = 1 − 0.5/p` çözücüsünü kullanın — `expected` ve
`thresholded` modlarının **ikisi de** MAE altında suboptimaldir:
```python
merdiven = fit_quantile_ladder(train[FEATURES], y, folds)
tahmin = conditional_quantile_from_hurdle(sonuc.oof_probability,
                                          {q: r.oof_predictions for q, r in merdiven.items()})
```

**6 · Hiperparametre araması** — objective'i de arama uzayına koyun
```python
tuned = tune_with_optuna(train[FEATURES], y, folds, metric=METRIC,
                         timeout=3600, search_objective=True)
print(tuned.objective_comparison())
```

**7 · Model zoo + harman**
```python
zoo = make_model_zoo(train[FEATURES], y, folds, metric=METRIC, test=test[FEATURES])
secilen = prune_by_correlation(zoo.oof_matrix, y, max_members=5)
agirliklar = hill_climb_weights({k: zoo.oof_matrix[k] for k in secilen}, y, metric=METRIC)
stack = stack_oof(zoo.oof_matrix, y, folds, test_predictions=zoo.test_matrix)
```
`stack_oof` hem stacking hem hill climbing skorunu raporlar. Fark küçükse
**hill climbing'i tercih edin** — jüri notebook'u okuyacak, açıklanabilirlik değerli.

**8 · Feature eleme**
```python
temiz = null_importance_filter(train[FEATURES], y)          # dakikalar
secim = shap_backward_selection(train[temiz["keep"]], y, folds)  # saatler
```

**9 · Son gün: çok tohumlu tam veri refit**
```python
tur = estimate_full_data_rounds(extract_best_iterations(result.models), n_folds=len(folds))
final = multi_seed_refit(train[FINAL], y, test[FINAL], params=tuned.best_params,
                         n_estimators=tur, seeds=range(15))
```

## Jüri çıktıları

Bunları **son gün üretmeye kalkmayın** — pipeline'ın parçası olmalı.
Değerlendirmenin üçte ikisi notebook + sunum.

In [ ]:
from gridup.reporting import (
    business_impact, cv_fold_table, error_by_segment,
    feature_importance_table, model_footprint,
    plot_error_by_segment, plot_fold_scores, plot_prediction_timeline,
)

# 1 · Fold tablosu — kararlılığı gösterir
display(cv_fold_table(result))
plot_fold_scores(result); plt.show()

# 2 · Model NEREDE yanılıyor — sunumun en ikna edici bölümü
segment_hatasi = error_by_segment(y_true, y_pred, test[GROUP_COLUMN], metric=METRIC)
plot_error_by_segment(segment_hatasi, metric=METRIC); plt.show()

# 3 · Sinyal nereden geliyor — 400 satırlık önem listesi yerine aile dağılımı
display(feature_importance_table(result, group_prefixes=(
    "tarih_", "tatil_", "komsu_", "bolge_", "sebep_")))

# 4 · Operasyonel maliyet — jüri bunu soruyor, modeli gerçekten çalıştıracak
print(model_footprint(result.models, elapsed_seconds=result.elapsed_seconds))

# 5 · İş dili — "MAE 2.95" değil, "ortalama 3 kesinti hatayla tahmin ediyoruz"
print(business_impact(y_true, y_pred, unit_label="kesinti")["ozet"])

## Sunum için not

Jüri koltuğunda **mühendisler ve iş birimleri** var, akademisyen değil.
2024 birincisinin sunumunun son üç slaydı tamamen iş değeriydi: açıklanabilir
çözüm, daraltılmış feature seti, ~25 MB model, yeni veriyle eğitilebilirlik.

Skor ilk 10'a sokar; bu bölüm ödülü belirler.